In [ ]:
# --- portable setup (added 2026-09-12 when the project moved to GitHub) ---------------------
# All paths below resolve from HME_ROOT, the repository root.  On Colab: mount Drive and point
# HME_ROOT at your clone.  Locally: run jupyter from the repo, or export HME_ROOT=/path/to/repo.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    HME_ROOT = os.environ.get('HME_ROOT', '/content/drive/MyDrive/hyperbolic-icd10')   # <-- edit
except ImportError:
    HME_ROOT = os.environ.get('HME_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
assert os.path.isdir(os.path.join(HME_ROOT, 'data')), f'HME_ROOT={HME_ROOT!r} is not the repo root'
print('HME_ROOT =', HME_ROOT)


# Phase 4 — Fair Comparison (REVISED): Each Method at Its Best + Ambitious Swings

**What changed from v2:**
- **Euclidean now uses its OWN native recipe** (Adam, no hierarchy loss, no ROOT freeze) — the hyperbolic machinery was sabotaging it (drove d=10 to 0.19 / rank 12k when its true performance is ~0.78 / rank ~8). Fair means *each method at its best*, not identical foreign machinery.
- **Const & position-dependent** keep the validated hyperbolic trainer (freeze, hierarchy, burnin) — the right tool for them.
- **Forced-graded** is now a proper grid row (theory law kappa=(log(1+b))^2, frozen).
- **NEW ambitious swing:** graded curvature + hard negatives, to push MAP *and* mean rank together.

The grid isolates geometry with each method trained the way that suits it. Metrics: MAP, mean rank, distortion. Dims 2/5/10.

## 1. Setup + load

In [1]:
!pip install geoopt
import os, pickle, numpy as np, torch, torch.nn as nn, geoopt, json
from collections import defaultdict, deque

ROOT_DIR=HME_ROOT
DATA_DIR=os.path.join(ROOT_DIR,'data/processed'); RESULTS_DIR=os.path.join(ROOT_DIR,'results')
OUT=os.path.join(RESULTS_DIR,'phase4_revised'); os.makedirs(OUT,exist_ok=True)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device',device)

with open(os.path.join(DATA_DIR,'icd10_tree_with_features.pkl'),'rb') as f: data=pickle.load(f)
nodes=data['nodes']; edges=data['edges']; features=data['features']
codes=list(nodes.keys()); code_to_idx={c:i for i,c in enumerate(codes)}; idx_to_code={i:c for c,i in code_to_idx.items()}
N=len(codes); edges_idx=[(code_to_idx[p],code_to_idx[c]) for p,c in edges]
kids_idx=defaultdict(list); parent_of={}
for u,v in edges_idx: kids_idx[u].append(v); parent_of[v]=u
ROOT=code_to_idx['ROOT']
connected=set(edges_idx)|set((v,u) for u,v in edges_idx)
print(f'N={N}, edges={len(edges_idx)}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.5 MB/s eta 0:00:00
device cuda
N=46817, edges=46816


## 2. Shared eval — MAP, mean rank, distortion (full eval, matches historical)

In [2]:
nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
rng=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in rng.choice(len(edges_idx),size=2000,replace=False)]

def d_euclid(a,allp): return np.linalg.norm(allp-a,axis=1)
def d_poincare(a,allp):
    diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))

def evaluate(pos,dist_fn):
    if np.isnan(pos).any(): return float('nan'),float('nan')
    ranks=[]; aps=[]
    for (u,v) in EVAL:
        d=dist_fn(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==v)[0][0])+1)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)),float(np.mean(ranks))

adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
rng=np.random.default_rng(0); DPAIRS=[]
for s in rng.integers(0,N,size=2000):
    dd={int(s):0}; q=deque([int(s)])
    while q:
        x=q.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; q.append(y)
    t=int(rng.integers(0,N))
    if t in dd and t!=s: DPAIRS.append((int(s),t,dd[t]))
def distortion(pos,dist_fn):
    if np.isnan(pos).any(): return float('nan')
    return float(np.mean([abs(float(dist_fn(pos[u],pos[v:v+1])[0])-dg)/dg for (u,v,dg) in DPAIRS]))

def report(pos,dist_fn,label):
    m,mr=evaluate(pos,dist_fn); di=distortion(pos,dist_fn)
    print(f'  {label:<20} MAP {m:.4f}  meanR {mr:8.0f}  distortion {di:.4f}')
    return {'MAP':round(m,4),'mean_rank':round(mr,1),'distortion':round(di,4)}
print('eval ready')

eval ready


## 3. Validated Phase 2 Poincaré components (verbatim) — for const & posdep

In [3]:
class PoincareEmbedding(nn.Module):
    """
    Poincaré ball embedding model with constant curvature K = -1.

    Stores N embedding vectors that live inside the open unit ball in d dimensions.
    The Poincaré metric makes distances explode near the unit sphere, which gives
    hyperbolic space its exponential volume growth — the property that lets
    low-dimensional embeddings capture tree structure efficiently.
    """

    def __init__(self, num_nodes, dim, init_scale=0.001):
        super().__init__()

        # The Poincaré ball manifold from geoopt.
        # The 'c' parameter is the absolute value of curvature: c=1 means K=-1.
        # Internally, geoopt uses negative-curvature conventions consistent with
        # Nickel-Kiela's original paper.
        self.manifold = geoopt.PoincareBall(c=1.0)

        # The embedding table, but as a ManifoldParameter that geoopt knows
        # is constrained to live on the Poincaré ball manifold. This is what
        # tells the Riemannian optimizer to respect the unit-ball constraint.
        self.embeddings = geoopt.ManifoldParameter(
            torch.empty(num_nodes, dim),
            manifold=self.manifold
        )

        # Initialize embeddings to small random values near the origin.
        # Near-origin initialization is critical for hyperbolic embeddings:
        # the metric becomes singular at the boundary, so starting close to
        # the boundary causes numerical instability.
        # The .data accessor lets us modify the parameter in-place without
        # tracking gradients.
        with torch.no_grad():
            self.embeddings.data.uniform_(-init_scale, init_scale)

        self.dim = dim
        self.num_nodes = num_nodes

    def forward(self, indices):
        """
        Look up embeddings for a batch of indices.

        Parameters
        ----------
        indices : torch.LongTensor of any shape
            Integer indices into the embedding table.

        Returns
        -------
        embeddings : torch.FloatTensor
            One trailing dimension of size self.dim added to the input shape.
        """
        # Index into the embedding table along axis 0
        return self.embeddings[indices]

    def distance(self, u, v):
        """
        Poincaré distance between two batches of embeddings.

        Parameters
        ----------
        u, v : torch.FloatTensor of shape [..., dim]
            Two batches of points in the Poincaré ball.

        Returns
        -------
        d : torch.FloatTensor of shape [...]
            Hyperbolic distance, one per pair.
        """
        # geoopt's manifold object provides the correct distance function.
        # Internally this computes d(u,v) = arcosh(1 + 2||u-v||^2 / [(1-||u||^2)(1-||v||^2)])
        return self.manifold.dist(u, v, keepdim=False)

def compute_loss_with_hierarchy(
    anchor_emb, positive_emb, negative_embs, distance_fn,
    margin=0.05, lambda_h=1.0,
):
    """Nickel-Kiela softmax loss + hierarchy regularization."""
    # Standard NK loss
    pos_dist = distance_fn(anchor_emb, positive_emb)
    neg_dist = distance_fn(anchor_emb.unsqueeze(1), negative_embs)
    all_dist = torch.cat([pos_dist.unsqueeze(1), neg_dist], dim=1)
    logsumexp_term = torch.logsumexp(-all_dist, dim=1)
    nk_per_anchor = pos_dist + logsumexp_term

    # Hierarchy regularization: penalize parent norm > child norm - margin
    anchor_norms = torch.norm(anchor_emb, dim=-1)
    positive_norms = torch.norm(positive_emb, dim=-1)
    hierarchy_violation = torch.relu(anchor_norms - positive_norms + margin)

    return nk_per_anchor.mean() + lambda_h * hierarchy_violation.mean()

def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """One training epoch with optional gradient-zero freeze on specific rows."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        # Zero out gradient rows for frozen indices (e.g., ROOT)
        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)

def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """train_one_epoch_with_freeze, but using compute_loss_with_hierarchy."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)


def train_poincare_with_freeze_and_hierarchy(
    edges_idx, N, connected, code_to_idx,
    dim=10,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    batch_size=1024,
    num_negatives=50,
    learning_rate=10.0,
    burnin_multiplier=0.01,
    init_scale=0.001,
    margin=0.05,
    lambda_h=1.0,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("=" * 60)
    print(f"🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training")
    print(f"   ROOT idx = {code_to_idx['ROOT']}, pinned for {burnin_epochs + freeze_train_epochs} epochs")
    print(f"   hierarchy: margin={margin}, lambda_h={lambda_h}")
    print(f"   lr={learning_rate}, num_negatives={num_negatives}")
    print("=" * 60)

    model = PoincareEmbedding(num_nodes=N, dim=dim, init_scale=init_scale).to(device)

    root_idx = code_to_idx['ROOT']
    with torch.no_grad():
        model.embeddings.data[root_idx].zero_()

    with torch.no_grad():
        root_norm_at_init = torch.norm(model.embeddings.data[root_idx]).item()
        print(f"After init: ROOT_norm = {root_norm_at_init:.6f}   (must be 0.000000)")

    optimizer = geoopt.optim.RiemannianSGD(model.parameters(), lr=learning_rate)

    loss_history = []
    freeze_until_epoch = burnin_epochs + freeze_train_epochs

    for epoch in range(num_epochs):
        effective_lr = (learning_rate * burnin_multiplier
                        if epoch < burnin_epochs else learning_rate)
        for pg in optimizer.param_groups:
            pg['lr'] = effective_lr

        freeze_indices = [root_idx] if epoch < freeze_until_epoch else None

        avg_loss = train_one_epoch_with_freeze_and_hierarchy(
            model, edges_idx, optimizer,
            batch_size, num_negatives, N, connected, device,
            margin=margin, lambda_h=lambda_h,
            freeze_indices=freeze_indices,
        )
        loss_history.append(avg_loss)

        if epoch < burnin_epochs:
            phase = "burn-in+freeze"
        elif epoch < freeze_until_epoch:
            phase = "FROZEN+hier"
        else:
            phase = "free+hier"

        if (epoch < 3 or epoch == burnin_epochs or epoch == freeze_until_epoch
                or (epoch + 1) % 10 == 0):
            with torch.no_grad():
                norms = torch.norm(model.embeddings.data, dim=-1)
                print(
                    f"Epoch {epoch+1:3d}/{num_epochs} [{phase}, lr={effective_lr:.4f}]: "
                    f"loss={avg_loss:.4f}, "
                    f"ROOT_norm={norms[code_to_idx['ROOT']].item():.4f}, "
                    f"mean_norm={norms.mean().item():.4f}, "
                    f"max_norm={norms.max().item():.4f}"
                )

    return model, loss_history



## 4. kappa features + shared helpers

In [4]:
kr=np.zeros((N,2),dtype=np.float32)
for cs,i in code_to_idx.items():
    f=features[cs]; kr[i,0]=np.log1p(f['branching_factor'])**2; kr[i,1]=np.log1p(f['subtree_size'])
kappa_raw=torch.tensor(kr,device=device)

# frozen graded law kappa = (log(1+b))^2 scaled to [-1,1]
bf_all=np.array([features[idx_to_code[i]]['branching_factor'] for i in range(N)])
kap_graded_np=np.log1p(bf_all)**2
kap_graded_np=2*(kap_graded_np-kap_graded_np.min())/(kap_graded_np.max()-kap_graded_np.min()+1e-9)-1
kap_graded=torch.tensor(kap_graded_np,dtype=torch.float32,device=device)

def sample_negs(anchors,K):
    out=np.random.randint(0,N,size=(len(anchors),K))
    for i,a in enumerate(anchors):
        for j in range(K):
            while out[i,j]==a or (a,out[i,j]) in connected: out[i,j]=np.random.randint(0,N)
    return out
print('features + helpers ready; kappa_raw',kappa_raw.shape)

features + helpers ready; kappa_raw torch.Size([46817, 2])


## 5. EUCLIDEAN — its own native recipe (NO hyperbolic machinery)

Adam, no hierarchy loss, no ROOT freeze, no burnin. This is the recipe that historically gave ~0.78 MAP / rank ~8 at d=10. Forcing hyperbolic machinery onto Euclidean sabotages it, so we don't.

In [5]:
def train_euclid_native(dim, epochs=50, K=50, lr=0.1, init_scale=1e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    emb=nn.Parameter(torch.empty(N,dim,device=device).uniform_(-init_scale,init_scale))
    opt=torch.optim.Adam([emb],lr=lr); E=np.array(edges_idx)
    for ep in range(epochs):
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=emb[a]; ep_=emb[p]; en=emb[ng]
            dp=((ea-ep_)**2).sum(-1); dn=((ea.unsqueeze(1)-en)**2).sum(-1)
            loss=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return emb.detach().cpu().numpy()
print('euclid native trainer ready')

euclid native trainer ready


## 6. HYPERBOLIC trainer (const / posdep / forced-graded / graded+hardneg)

Uses the validated PoincareEmbedding + freeze + hierarchy + burnin. `mode` selects the curvature:
- `const`: alpha=0 (no curvature)
- `learned`: learnable field kappa=kappa_scale*tanh(W.feat), alpha=-0.8
- `graded`: FROZEN theory law kappa=(log(1+b))^2, alpha=-0.8
- `graded_hardneg`: graded + hard negative mining (attacks MAP and rank together)

In [6]:
def train_hyper(mode, dim, alpha=-0.8, epochs=200, K=50, lr=10.0,
                burnin=10, burnin_mult=0.01, margin=0.05, lambda_h=1.0,
                init_scale=1e-3, kappa_scale=1.0, hard_frac=0.5, hard_pool=200, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,init_scale).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    W=None; fopt=None
    if mode=='learned':
        W=nn.Parameter(torch.zeros(2,1,device=device)); fopt=torch.optim.Adam([W],lr=0.2,weight_decay=1e-4)
    def node_kappa():
        if mode=='const': return torch.zeros(N,device=device)
        if mode=='learned': return kappa_scale*torch.tanh(kappa_raw@W).squeeze(-1)
        return kap_graded   # graded / graded_hardneg
    def cdist(u,v,ku,kv):
        base=model.manifold.dist(u,v)
        if mode=='const': return base
        return base*torch.exp(alpha*0.5*(ku+kv))
    def hard_negs(anchors,kap):
        nh=int(K*hard_frac); nr=K-nh; out=np.zeros((len(anchors),K),dtype=np.int64)
        emb=model.embeddings.detach()
        for i,a in enumerate(anchors):
            cand=np.random.randint(0,N,size=hard_pool); cand=cand[cand!=a]
            cand=np.array([c for c in cand if (a,c) not in connected])
            if len(cand)<nh:
                out[i]=np.random.randint(0,N,size=K); continue
            with torch.no_grad():
                ea=emb[a].unsqueeze(0); ct=torch.tensor(cand,device=device)
                d=model.manifold.dist(ea,emb[ct])
                if mode!='const': d=d*torch.exp(alpha*0.5*(kap[a]+kap[ct]))
                hard=ct[torch.argsort(d)[:nh]].cpu().numpy()
            rand=np.random.randint(0,N,size=nr)
            out[i]=np.concatenate([hard,rand])
        return out
    E=np.array(edges_idx)
    for ep in range(epochs):
        eff=lr*burnin_mult if ep<burnin else lr
        for pg in opt.param_groups: pg['lr']=eff
        kap=node_kappa()
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            if mode=='graded_hardneg' and ep>=burnin:
                ng=torch.tensor(hard_negs(b[:,0],kap)).to(device)
            else:
                ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            ka=kap[a]; kp=kap[p]; kn=kap[ng]
            dp=cdist(ea,ep_,ka,kp)
            dn=cdist(ea.unsqueeze(1),en,ka.unsqueeze(1),kn)
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+lambda_h*torch.relu(an-pn+margin).mean()
            opt.zero_grad()
            if fopt: fopt.zero_grad()
            loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
            if fopt: fopt.step()
            kap=node_kappa()
    pos=model.embeddings.detach().cpu().numpy()
    return pos, node_kappa().detach().cpu().numpy()
print('hyperbolic trainer ready')

hyperbolic trainer ready


## 7. Run the full grid at d=2, 5, 10

In [7]:
def kappa_ramp(kap):
    bins=sorted(set(int(b) for b in bf_all if 1<=b<=15)); means=[float(kap[bf_all==b].mean()) for b in bins]
    return float(np.corrcoef(bins,means)[0,1]) if len(bins)>2 else 0.0

grid={}
for dim in [2,5,10]:
    print(f'\n===== dim {dim} =====')
    pos=train_euclid_native(dim);                 grid[('euclid',dim)]=report(pos,d_euclid,'euclid (native)')
    pos,_=train_hyper('const',dim);               grid[('const',dim)]=report(pos,d_poincare,'const a=0')
    pos,k=train_hyper('learned',dim);             r=report(pos,d_poincare,'posdep learned'); r['ramp']=round(kappa_ramp(k),3); grid[('posdep',dim)]=r
    pos,k=train_hyper('graded',dim);              grid[('graded',dim)]=report(pos,d_poincare,'forced-graded')
    pos,k=train_hyper('graded_hardneg',dim);      grid[('graded_hn',dim)]=report(pos,d_poincare,'graded+hardneg')


===== dim 2 =====
  euclid (native)      MAP 0.0419  meanR      464  distortion 0.3839
  const a=0            MAP 0.5630  meanR     1375  distortion 0.7501
  posdep learned       MAP 0.4852  meanR      768  distortion 0.6946
  forced-graded        MAP 0.5478  meanR     1258  distortion 0.6936
  graded+hardneg       MAP 0.4268  meanR     1143  distortion 0.7818

===== dim 5 =====
  euclid (native)      MAP 0.2145  meanR      116  distortion 0.2699
  const a=0            MAP 0.6754  meanR     1055  distortion 0.6307
  posdep learned       MAP 0.7031  meanR      478  distortion 0.6121
  forced-graded        MAP 0.7082  meanR     1020  distortion 0.6118
  graded+hardneg       MAP 0.7096  meanR      811  distortion 0.5916

===== dim 10 =====
  euclid (native)      MAP 0.4809  meanR       44  distortion 0.5959
  const a=0            MAP 0.7244  meanR      572  distortion 0.6106
  posdep learned       MAP 0.7504  meanR      190  distortion 0.6121
  forced-graded        MAP 0.7573  meanR     

## 8. Grid summary — MAP and mean rank

In [8]:
methods=[('euclid','Euclid'),('const','Const'),('posdep','PosDep'),('graded','Graded'),('graded_hn','Graded+HN')]
print('MAP:')
print(f"{'dim':>4}"+''.join(f'{lbl:>11}' for _,lbl in methods))
for dim in [2,5,10]:
    print(f'{dim:>4}'+''.join(f"{grid[(m,dim)]['MAP']:>11.4f}" for m,_ in methods))
print('\nMEAN RANK:')
print(f"{'dim':>4}"+''.join(f'{lbl:>11}' for _,lbl in methods))
for dim in [2,5,10]:
    print(f'{dim:>4}'+''.join(f"{grid[(m,dim)]['mean_rank']:>11.0f}" for m,_ in methods))
print('\nBEST per dim (MAP / rank):')
for dim in [2,5,10]:
    bmap=max(methods,key=lambda x:grid[(x[0],dim)]['MAP']); brank=min(methods,key=lambda x:grid[(x[0],dim)]['mean_rank'])
    print(f'  d={dim}: best MAP = {bmap[1]} ({grid[(bmap[0],dim)]["MAP"]:.4f}), best rank = {brank[1]} ({grid[(brank[0],dim)]["mean_rank"]:.0f})')
with open(os.path.join(OUT,'phase4_revised_grid.json'),'w') as f:
    json.dump({str(k):v for k,v in grid.items()},f,indent=2,default=str)
print('saved phase4_revised_grid.json')

MAP:
 dim     Euclid      Const     PosDep     Graded  Graded+HN
   2     0.0419     0.5630     0.4852     0.5478     0.4268
   5     0.2145     0.6754     0.7031     0.7082     0.7096
  10     0.4809     0.7244     0.7504     0.7573     0.7503

MEAN RANK:
 dim     Euclid      Const     PosDep     Graded  Graded+HN
   2        464       1375        768       1258       1143
   5        116       1055        478       1020        811
  10         44        572        190        550        516

BEST per dim (MAP / rank):
  d=2: best MAP = Const (0.5630), best rank = Euclid (464)
  d=5: best MAP = Graded+HN (0.7096), best rank = Euclid (116)
  d=10: best MAP = Graded (0.7573), best rank = Euclid (44)
saved phase4_revised_grid.json


## 9. How to read this

- **Euclidean** is now its true self (native recipe) — expect ~0.78 / rank ~8 at d=10, matching history. If it's not, the harness still has an issue.
- **Graded** should beat **PosDep** on MAP (the forced-graded win you found).
- **Graded+HN** is the ambitious swing: graded curvature (fixes MAP) + hard negatives (fixes rank). Watch whether it gets BOTH the best MAP and a much lower mean rank than graded alone.
- The 'BEST per dim' line tells you, at each dimension, which method wins MAP and which wins rank. The goal is a single method (ideally Graded+HN) winning both.

If Graded+HN doesn't crush the rank, the next lever is a curvature term targeting the boundary shell specifically — but hard negatives are the standard fix for exactly the 'good MAP, bad tail' pattern.